# Day 10: Async JavaScript and Node.js
**Promises, async/await, npm, Express CRUD APIs and testing with Postman**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
Google Colab already has Node.js and npm. In each JavaScript section, `%%writefile` saves a file and `!node` runs it.

**VS Code:** install Node.js LTS from nodejs.org, then run the same files with `node file.js` in the terminal.

In [ ]:
# ===== Option A: Google Colab (active) =====
!node --version
!npm --version

# ===== Option B: VS Code =====
# In the VS Code terminal:
#   node --version
#   npm --version

## 2. The event loop: predict, then run

In [ ]:
%%writefile order.js
console.log("1: order taken");
setTimeout(() => console.log("4: coffee ready (timer 100 ms)"), 100);
setTimeout(() => console.log("3: tea ready (timer 0 ms)"), 0);
console.log("2: next customer");

In [ ]:
# ===== Option A: Google Colab (active) =====
!node order.js

# ===== Option B: VS Code =====
# node order.js

## 3. Promises

In [ ]:
%%writefile promises.js
function cook(dish, ms = 300) {
  return new Promise((resolve, reject) => {
    setTimeout(() => {
      if (dish === "pizza") reject(new Error(`out of cheese for ${dish}`));
      else resolve(`${dish} ready`);
    }, ms);
  });
}

cook("rice")
  .then(msg => { console.log(msg); return cook("curry"); })
  .then(msg => { console.log(msg); return cook("pizza"); })
  .then(msg => console.log(msg))
  .catch(err => console.log("Error:", err.message))
  .finally(() => console.log("Kitchen closed"));

module.exports = { cook };

In [ ]:
# ===== Option A: Google Colab (active) =====
!node promises.js

# ===== Option B: VS Code =====
# node promises.js

## 4. async / await and Promise.all

In [ ]:
%%writefile async_demo.js
const wait = (ms) => new Promise(resolve => setTimeout(resolve, ms));
const cook = async (dish) => { await wait(500); return `${dish} ready`; };

async function main() {
  let start = Date.now();
  await cook("rice");
  await cook("curry");
  await cook("dal");
  console.log(`Sequential: ${Date.now() - start} ms`);

  start = Date.now();
  const dishes = await Promise.all([cook("rice"), cook("curry"), cook("dal")]);
  console.log(`Parallel:   ${Date.now() - start} ms`, dishes);

  try {
    await Promise.reject(new Error("burnt toast"));
  } catch (err) {
    console.log("Caught:", err.message);
  }
}
main();

In [ ]:
# ===== Option A: Google Colab (active) =====
!node async_demo.js

# ===== Option B: VS Code =====
# node async_demo.js

### TODO 1: Use `Promise.allSettled` with one dish that fails, and print the status of each result.

In [ ]:
# TODO: write your code here (use %%writefile and !node)

## 5. npm and Express

In [ ]:
# ===== Option A: Google Colab (active) =====
!npm init -y > /dev/null
!npm install express --silent
!npm ls express

# ===== Option B: VS Code =====
# In your project folder terminal:
#   npm init -y
#   npm install express

In [ ]:
%%writefile server.js
const express = require("express");
const app = express();
app.use(express.json());

// Logging middleware: method, path, status, time
app.use((req, res, next) => {
  const start = Date.now();
  res.on("finish", () => console.log(`${req.method} ${req.originalUrl} ${res.statusCode} ${Date.now() - start}ms`));
  next();
});

let tasks = [];
let nextId = 1;
const findTask = (id) => tasks.find(t => t.id === Number(id));

app.get("/health", (req, res) => res.json({ status: "ok" }));

app.get("/tasks", (req, res) => {
  const { done } = req.query;
  if (done === undefined) return res.json(tasks);
  res.json(tasks.filter(t => String(t.done) === done));
});

app.get("/tasks/:id", (req, res) => {
  const task = findTask(req.params.id);
  if (!task) return res.status(404).json({ error: "Task not found" });
  res.json(task);
});

app.post("/tasks", (req, res) => {
  const { title } = req.body;
  if (!title) return res.status(400).json({ error: "title is required" });
  const task = { id: nextId++, title, done: false };
  tasks.push(task);
  res.status(201).json(task);
});

app.put("/tasks/:id", (req, res) => {
  const task = findTask(req.params.id);
  if (!task) return res.status(404).json({ error: "Task not found" });
  Object.assign(task, { title: req.body.title ?? task.title, done: req.body.done ?? task.done });
  res.json(task);
});

app.delete("/tasks/:id", (req, res) => {
  const task = findTask(req.params.id);
  if (!task) return res.status(404).json({ error: "Task not found" });
  tasks = tasks.filter(t => t !== task);
  res.status(204).end();
});

app.use((req, res) => res.status(404).json({ error: "Route not found" }));

const PORT = process.env.PORT || 3000;
app.listen(PORT, () => console.log(`Tasks API on http://localhost:${PORT}`));

## 6. An async client that calls the API

In [ ]:
%%writefile client.js
const BASE = process.env.BASE_URL || "http://localhost:3000";

async function api(method, path, body) {
  const res = await fetch(BASE + path, {
    method,
    headers: { "Content-Type": "application/json" },
    body: body ? JSON.stringify(body) : undefined,
  });
  if (res.status === 204) return null;
  const data = await res.json();
  if (!res.ok) throw new Error(`HTTP ${res.status}: ${data.error}`);
  return data;
}

async function main() {
  const created = await Promise.all(
    ["Read post-read", "Finish workbook", "Install Postman"].map(title => api("POST", "/tasks", { title }))
  );
  console.log("Created", created.length, "tasks");
  await api("PUT", `/tasks/${created[0].id}`, { done: true });
  await api("DELETE", `/tasks/${created[1].id}`);
  console.log("Done only:", await api("GET", "/tasks?done=true"));
  console.log("Final:", await api("GET", "/tasks"));
  try {
    await api("POST", "/tasks", {});
  } catch (err) {
    console.log("Expected error ->", err.message);
  }
}
main();

Start the server in the background, run the client, then stop the server.

**VS Code:** open two terminals. Run `node server.js` in the first and `node client.js` in the second. Stop the server with `Ctrl + C`.

In [ ]:
# ===== Option A: Google Colab (active) =====
!nohup node server.js > server.log 2>&1 & echo $! > server.pid
!sleep 2
!node client.js
!kill $(cat server.pid)
!cat server.log

# ===== Option B: VS Code =====
# Terminal 1: node server.js
# Terminal 2: node client.js

## 7. A ready-made Postman collection
This cell writes `tasks_api.postman_collection.json`. In Postman: **Import**, choose the file, then set the collection variable `baseUrl` to `http://localhost:3000`.

In [ ]:
import json

def req(name, method, path, body=None, status=200):
    item = {"name": name, "request": {"method": method, "header": [{"key": "Content-Type", "value": "application/json"}],
            "url": {"raw": "{{baseUrl}}" + path, "host": ["{{baseUrl}}"], "path": [p for p in path.split("?")[0].split("/") if p]}},
            "event": [{"listen": "test", "script": {"type": "text/javascript",
                       "exec": [f'pm.test("status is {status}", () => pm.response.to.have.status({status}));']}}]}
    if body is not None:
        item["request"]["body"] = {"mode": "raw", "raw": json.dumps(body, indent=2)}
    return item

collection = {
    "info": {"name": "Day 10 Tasks API", "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"},
    "variable": [{"key": "baseUrl", "value": "http://localhost:3000"}],
    "item": [
        req("Health", "GET", "/health"),
        req("Create task", "POST", "/tasks", {"title": "Learn Express"}, 201),
        req("Create task without title", "POST", "/tasks", {}, 400),
        req("List tasks", "GET", "/tasks"),
        req("Get task 1", "GET", "/tasks/1"),
        req("Update task 1", "PUT", "/tasks/1", {"done": True}),
        req("Delete task 1", "DELETE", "/tasks/1", status=204),
        req("Get missing task", "GET", "/tasks/999", status=404),
    ],
}
with open("tasks_api.postman_collection.json", "w") as f:
    json.dump(collection, f, indent=2)
print("Saved with", len(collection["item"]), "requests")

# ===== Option A: Google Colab, download the file (uncomment to use) =====
# from google.colab import files
# files.download("tasks_api.postman_collection.json")

# ===== Option B: VS Code =====
# The file is already in your project folder.

### TODO 2: Add `GET /tasks/stats` returning total, done and pending counts. Place it above `/tasks/:id` so it matches first, then add it to the Postman collection.

In [ ]:
# TODO: write your code here

---
**Yoi shuumatsu wo** (良い週末を): *Have a great weekend*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)